# 03 — SigLIP-B/16-384 Frozen Baseline (B0)

Establishes the frozen SigLIP segmentation baseline on VOC 2012 val.  
Run **once**; record results in `report.md`.

- Model: `google/siglip-base-patch16-384` (no fine-tuning, no LoRA)
- Eval: MaskCLIP-style last-attention bypass, zero-shot on VOC 2012 val
- Includes a τ_seg sweep to find the best threshold for this model

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'datasets', 'transformers>=4.40', 'huggingface_hub', 'tqdm', 'Pillow',
], check=True)
print('Dependencies installed.')

In [ ]:
import os, types, getpass
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torchvision.datasets as tvd

from transformers import SiglipModel, AutoProcessor

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID  = 'google/siglip-base-patch16-384'
EVAL_SIZE = 384
PATCH_SIZE = 16
N_SIDE    = EVAL_SIZE // PATCH_SIZE   # 24

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

TAU_CANDIDATES  = [-0.3, -0.2, -0.1, -0.05, 0.0, 0.05, 0.1]
TAU_SWEEP_N     = 100    # images for the sweep
FULL_EVAL_N     = 200    # images for the final number

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
processor = AutoProcessor.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
model     = SiglipModel.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])
model     = model.to(DEVICE).eval()

total = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_ID}')
print(f'Params: {total/1e6:.1f}M')

In [ ]:
# ── VOC 2012 val ──────────────────────────────────────────────────────────────
VOC_ROOT = Path('/tmp/voc')
VOC_ROOT.mkdir(exist_ok=True)
voc_val  = tvd.VOCSegmentation(root=str(VOC_ROOT), year='2012',
                                image_set='val', download=True)
print(f'VOC 2012 val: {len(voc_val)} images')

In [ ]:
# ── Eval function ─────────────────────────────────────────────────────────────

def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    return self.out_proj(self.v_proj(hidden_states)), None


def voc_eval(model, processor, n_images, tau_seg, device=DEVICE):
    """MaskCLIP-style zero-shot segmentation on VOC 2012 val."""
    model.eval()
    n_fg  = len(VOC_CLASSES)
    n_cls = n_fg + 1

    prompts = [f'a photo of a {c}' for c in VOC_CLASSES]
    txt_in  = processor(text=prompts, return_tensors='pt', padding=True).to(device)
    with torch.no_grad():
        txt_feats = F.normalize(
            model.text_model(**txt_in).last_hidden_state[:, -1, :], dim=-1
        )

    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    tp = torch.zeros(n_cls, dtype=torch.float64)
    fp = torch.zeros(n_cls, dtype=torch.float64)
    fn = torch.zeros(n_cls, dtype=torch.float64)
    gt_present = torch.zeros(n_cls, dtype=torch.bool)

    try:
        for i in tqdm(range(n_images), desc=f'VOC eval τ={tau_seg}', leave=False):
            pil_img, target = voc_val[i]
            orig_w, orig_h  = pil_img.size
            gt = torch.from_numpy(np.array(target)).long()

            pix = processor(images=pil_img, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)

            patch_n = F.normalize(out.last_hidden_state[0], dim=-1)
            sim     = patch_n @ txt_feats.T
            n_side  = int(sim.shape[0] ** 0.5)
            sim_up  = F.interpolate(
                sim.reshape(n_side, n_side, n_fg).permute(2,0,1).unsqueeze(0).float(),
                size=(orig_h, orig_w), mode='bilinear', align_corners=False
            ).squeeze(0).permute(1, 2, 0)

            max_sim, pred = sim_up.max(dim=-1)
            pred = pred + 1
            pred[max_sim < tau_seg] = 0
            pred = pred.cpu()

            valid = (gt != 255)
            pv, gv = pred[valid], gt[valid]
            for c in range(n_cls):
                pc = (pv == c); gc = (gv == c)
                tp[c] += (pc & gc).sum()
                fp[c] += (pc & ~gc).sum()
                fn[c] += (~pc & gc).sum()
                if gc.any(): gt_present[c] = True
    finally:
        last_attn.forward = orig_fwd

    iou  = tp / (tp + fp + fn).clamp(min=1e-6)
    miou = iou[gt_present].mean().item() * 100
    return miou, iou.cpu().numpy() * 100


print('voc_eval defined.')

In [ ]:
# ── τ sweep ───────────────────────────────────────────────────────────────────
print(f'Sweeping τ_seg on {TAU_SWEEP_N} images (MaskCLIP)...')
print(f'{"τ_seg":>8s}  {"mIoU (%)":>10s}')
print('-' * 22)

tau_results = {}
for tau in TAU_CANDIDATES:
    miou, _ = voc_eval(model, processor, TAU_SWEEP_N, tau)
    tau_results[tau] = miou
    print(f'{tau:8.2f}  {miou:10.2f}')

best_tau = max(tau_results, key=tau_results.get)
print(f'\nBest τ_seg = {best_tau}  (mIoU = {tau_results[best_tau]:.2f}%)')

In [ ]:
# ── Full B0 eval at best τ ────────────────────────────────────────────────────
b0_miou, b0_per_cls = voc_eval(model, processor, FULL_EVAL_N, best_tau)

print(f'\nB0 (frozen SigLIP-B/16-384)  τ={best_tau}  {FULL_EVAL_N} images')
print(f'mIoU: {b0_miou:.2f}%')

print('\nPer-class IoU:')
print(f'{"background":15s}: {b0_per_cls[0]:.1f}%')
for i, cls in enumerate(VOC_CLASSES):
    print(f'{cls:15s}: {b0_per_cls[i+1]:.1f}%')

In [ ]:
# ── Summary for report.md ─────────────────────────────────────────────────────
print('=== Copy this into report.md ===')
print(f'| B0 (frozen SigLIP-B/16-384) | τ={best_tau} | {FULL_EVAL_N} images | {b0_miou:.2f}% |')